In [7]:
import sys
import os
import json
import io
import warnings
import numpy as np
from pathlib import Path
from datetime import datetime
from contextlib import redirect_stdout, redirect_stderr

import torch
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from transformers.utils import logging as hf_logging
from huggingface_hub.utils import disable_progress_bars
from IPython.display import display, Markdown

project_root = Path.cwd()
if not (project_root / "config" / "settings.py").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from config.settings import (
    TEXT_CHUNKS_PATH,
    EMBEDDINGS_DIR,
    EMBEDDINGS_NPY_PATH,
    EMBEDDINGS_META_PATH,
    EMBEDDING_MODEL_NAME,
    OUTPUT_DIRS,
)

for d in OUTPUT_DIRS:
    d.mkdir(parents=True, exist_ok=True)

load_dotenv(project_root / ".env", override=True)

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token

warnings.filterwarnings("ignore", category=FutureWarning)
hf_logging.set_verbosity_error()
disable_progress_bars()

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

In [8]:
assert TEXT_CHUNKS_PATH.exists(), f"Not found: {TEXT_CHUNKS_PATH}"

chunks = json.loads(TEXT_CHUNKS_PATH.read_text(encoding="utf-8"))
texts = [chunk["text"] for chunk in chunks]

assert len(texts) == len(chunks), "Texts and chunks count mismatch"
assert all(isinstance(text, str) and text.strip() for text in texts), "Some chunks have empty text"

avg_chars = sum(len(text) for text in texts) / len(texts)

display(Markdown(f"""
### Chunks Loaded

| Item | Value |
|------|-------|
| **Total chunks** | {len(chunks)} |
| **Average text length** | {avg_chars:.0f} chars |
| **Source** | `{TEXT_CHUNKS_PATH.relative_to(project_root)}` |
""".strip()))

### Chunks Loaded

| Item | Value |
|------|-------|
| **Total chunks** | 1391 |
| **Average text length** | 2008 chars |
| **Source** | `data\text\text_chunks.json` |

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"

with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
    model = SentenceTransformer(
        EMBEDDING_MODEL_NAME,
        device=device,
        token=hf_token if hf_token else None,
    )

embedding_dim = model.get_embedding_dimension()

display(Markdown(f"""
### Embedding Model Loaded

| Item | Value |
|------|-------|
| **Model** | `{EMBEDDING_MODEL_NAME}` |
| **Embedding dimension** | {embedding_dim} |
| **Status** | Ready |
""".strip()))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### Embedding Model Loaded

| Item | Value |
|------|-------|
| **Model** | `sentence-transformers/all-mpnet-base-v2` |
| **Embedding dimension** | 768 |
| **Status** | Ready |

In [10]:
embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=64,
    normalize_embeddings=True,
)

embeddings = np.asarray(embeddings, dtype=np.float32)

assert embeddings.shape[0] == len(chunks), (
    f"Mismatch: {embeddings.shape[0]} embeddings vs {len(chunks)} chunks"
)

assert embeddings.shape[1] == embedding_dim, (
    f"Dimension mismatch: {embeddings.shape[1]} vs expected {embedding_dim}"
)

display(Markdown(f"""
### Embeddings Generated

| Item | Value |
|------|-------|
| **Chunks embedded** | {embeddings.shape[0]} |
| **Embedding dimension** | {embeddings.shape[1]} |
| **Normalized** | Yes |
| **Dtype** | `{embeddings.dtype}` |
| **Array size** | {embeddings.nbytes / (1024 * 1024):.2f} MB |
""".strip()))

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

### Embeddings Generated

| Item | Value |
|------|-------|
| **Chunks embedded** | 1391 |
| **Embedding dimension** | 768 |
| **Normalized** | Yes |
| **Dtype** | `float32` |
| **Array size** | 4.08 MB |

In [11]:
np.save(str(EMBEDDINGS_NPY_PATH), embeddings)

metadata = []

for chunk in chunks:
    metadata.append({
        "chunk_id": chunk["chunk_id"],
        "chunk_index": chunk["chunk_index"],
        "char_count": chunk["char_count"],
        "word_count": chunk["word_count"],
        "page_start": chunk["page_start"],
        "page_end": chunk["page_end"],
        "source_pages": chunk["source_pages"],
        "toc_level": chunk["toc_level"],
        "section_title": chunk["section_title"],
        "section_path": chunk["section_path"],
        "part_index": chunk["part_index"],
        "part_count": chunk["part_count"],
    })

EMBEDDINGS_META_PATH.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

display(Markdown(f"""
### Outputs Saved

| File | Path |
|------|------|
| **Embeddings** | `{EMBEDDINGS_NPY_PATH.relative_to(project_root)}` |
| **Metadata** | `{EMBEDDINGS_META_PATH.relative_to(project_root)}` |
""".strip()))

### Outputs Saved

| File | Path |
|------|------|
| **Embeddings** | `data\embeddings\chunk_embeddings.npy` |
| **Metadata** | `data\embeddings\chunk_embeddings_metadata.json` |

In [13]:
npy_size = EMBEDDINGS_NPY_PATH.stat().st_size / (1024 * 1024)
meta_size = EMBEDDINGS_META_PATH.stat().st_size / (1024 * 1024)

loaded_embeddings = np.load(str(EMBEDDINGS_NPY_PATH))

assert loaded_embeddings.shape == embeddings.shape, "Reload shape mismatch"
assert len(metadata) == embeddings.shape[0], "Metadata and embeddings count mismatch"

display(Markdown(f"""
### Embedding Generation Complete

| Item | Value |
|------|-------|
| **Model** | `{EMBEDDING_MODEL_NAME}` |
| **Total chunks embedded** | {embeddings.shape[0]} |
| **Embedding dimension** | {embeddings.shape[1]} |
| **Embeddings file size** | {npy_size:.2f} MB |
| **Metadata file size** | {meta_size:.2f} MB |
| **Normalized** | Yes |
| **Reload verified** | Yes |
| **Timestamp** | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} |
""".strip()))

### Embedding Generation Complete

| Item | Value |
|------|-------|
| **Model** | `sentence-transformers/all-mpnet-base-v2` |
| **Total chunks embedded** | 1391 |
| **Embedding dimension** | 768 |
| **Embeddings file size** | 4.08 MB |
| **Metadata file size** | 0.66 MB |
| **Normalized** | Yes |
| **Reload verified** | Yes |
| **Timestamp** | 2026-04-24 10:38:04 |